In [1]:
import sys
print("Using Python version:", sys.version)

Using Python version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


In [ ]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import csv
import pandas as pd
import xml.etree.ElementTree as ET 
from IPython.display import display, Markdown

import requests
from openai import OpenAI

In [4]:
## Configuration and Client set-up

# Defining the PuBMed webservice endpoints
PUBMED_SEARCH_URL = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi")  # search engine endpoint, which returns a list of PubMed IDs

# The data retreival endpoint, takes a list of PMIDs, 
# it returns the actual article metadata, including paper titles, publication dates, and full abstract texts in XML or text format
PUBMED_FETCH_URL = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi")    

NCBI_EMAIL = "manjunathrampure.co@gmail.com"

# Dynamically selecting the AI model
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

#Instantiates the primary OpenAI() client object from the openai SDK library, which manages connections, headers, and HTTP calls to OpenAI's servers.
client = OpenAI()

In [5]:
# converting the random question into a structured, Boolean-formatted search string that PubMed's search engine can understand.

def create_pubmed_query(question):
    """
    Ask the model to convert a question into a query.
    """
    response = client.responses.create(
        model = MODEL,
        instructions = (
            "Convert the research question into a consise PubMed query. "
            "Use biological keywords, Boolean operators, and synonyms where "
            "helpful. Return only the query"
        ),
        input = question,
    )

    # strip() - removes any leading or trailing whitespace, newlines (\n), or space padding
    return response.output_text.strip()

In [6]:
my_test_query = create_pubmed_query("Is CD19 a better target for B-cell Malignancies?")

In [7]:
print(my_test_query)

("CD19"[Title/Abstract] OR "cluster of differentiation 19"[Title/Abstract]) AND ("B-Cell Neoplasms"[MeSH Terms] OR "B-cell malignan*"[Title/Abstract] OR B-cell leukemia*[Title/Abstract] OR B-cell lymphoma*[Title/Abstract]) AND (target*[Title/Abstract] OR immunotherap*[Title/Abstract] OR "CAR-T"[Title/Abstract] OR "chimeric antigen receptor T-cell"[Title/Abstract])


In [8]:
## search pubmed 

def search_pubmed(query, start_year = 2021, maximum_results = 5):
    """
    Search PubMed and return the matching PubMed IDs
    """
    # parentheses to your (query) can group your search query together so that the filter applies to the entire query
    # pubmed acts as an database search engine, with special tags in [] 
    # Modification 1: Construct the date filter syntax
    date_filter = f'("{start_year}"[Date - Publication] : "3000"[Date - Publication])'
    search_query = f"({query}) AND {date_filter} AND hasabstract[Filter]"

    parameters = {
        "db": "pubmed",
        "term": search_query,
        "retmax": maximum_results,
        "retmode": "json",
        "sort": "relevance",
        "tool": "hackbio_pubmed_course", #NCBI usage requirements that identify your application and provide a contact email
        "email": NCBI_EMAIL, 
    }

    response = requests.get(
        PUBMED_SEARCH_URL,
        params = parameters,
        timeout = 30
    )
    
    response.raise_for_status()

    # parsing the returned JSON data and extracts the list of PubMed ID strings stored inside th esearch result
    return response.json()["esearchresult"]["idlist"]

In [9]:
# extracting the PMIDs by using the search_pubmed function
my_test_pids = search_pubmed(my_test_query)

In [10]:
# extracted PMIDs are
print(my_test_pids)

['35012754', '37055515', '38831163', '34923107', '39749335']


In [11]:
## Fetch abstract
# this block returns the pubmed ID numbers, into structured python dictionaries containing the title, publication year, abstract text, and direct web link for each article.


# defining a small helper function, which cleans up the text extracted from an XML element
def element_text(element):
    """
    Collect all text contained inside an XML element.
    """

    if element is None:
        return ""

    return " ".join("".join(element.itertext()).split())

# main function
def fetch_abstracts(pubmed_ids):
    """
    Retrieve article information for a list of PubMeb IDs.
    """

    # input check and parameter setup
    if not pubmed_ids:
        return []

    parameters = {
        "db": "pubmed",
        "id": ",".join(pubmed_ids),
        "retmode": "xml",
        "tool": "hackbio_pubmed_course",
        "email": NCBI_EMAIL,
    }

    # make the HTTP request and parse the XML
    response = requests.get(
        PUBMED_FETCH_URL,
        params = parameters,
        timeout = 30
    )

    # the returned XML response is parsed into an ElementTree root node(root) so Python can search through XML tags
    root = ET.fromstring(response.content)
    articles = []

    # loop through articles and extract the fields
    for record in root.findall(".//PubmedArticle"):
        citation = record.find("MedlineCitation")
        article = citation.find("Article")

        # extracting title and PMIDs
        pmid = citation.findtext("PMID", default = "")
        title = element_text(article.find("ArticleTitle"))

        # reconstructing structured abstracts
        # scientific abstracts are often split into labeled sections(BACKGROUND, METHODS, RESULTS, CONCLUSION)
        abstract_parts = []
        for section in article.findall("./Abstract/AbstractText"):
            text = element_text(section)
            label = section.attrib.get("Label")

            if label and text:
                abstract_parts.append(f"{label}: {text}")
            elif text:
                abstract_parts.append(text)

        abstract = " ".join(abstract_parts)

        # extracting the Publication Year with Fallbacks
        year = (
            article.findtext("./Journal/JournalIssue/PubDate/Year")
            or article.findtext("./ArticleDate/Year")
            or "Year not available"
        )

        # Build the result dictionary
        if abstract:
            articles.append(
                {
                    "pmid": pmid,
                    "title": title,
                    "year": year,
                    "abstract": abstract,
                    "link": f"https://pubmed.ncbi.nlm.nih.gov./{pmid}",
                }
            )

    return articles

In [12]:
# fetching the abstracts from the given PMIDs 
my_test_articles = fetch_abstracts(my_test_pids)

In [13]:
# extracted abstracts are
print(my_test_articles)

[{'pmid': '35012754', 'title': 'CAR T-cell therapy for B-cell lymphoma.', 'year': '2022', 'abstract': "Chimeric antigen receptor-modified (CAR) T-cell therapy targeting CD19 has revolutionized the treatment of relapsed or refractory B-cell lymphomas. Based on unprecedented response rates and durability of response in high risk B-cell lymphoma patients, anti-CD19 CAR T-cell therapy was rapidly approved by the FDA for a variety of lymphoma subtypes. Anti-CD19 CAR T-cell therapy is now considered standard of care for patients with relapsed or refractory (R/R) aggressive non-Hodgkin's Lymphoma (NHL) after 2 or more lines of therapy. Three second-generation anti-CD19 CAR T-cell products have been FDA approved for R/R aggressive B-cell lymphoma and FDA approval has been obtained for Mantle Cell Lymphoma and Follicular lymphoma as well. This has ensured broad access to CAR T-cell therapy for patients with NHL and new real-world trials have helped confirm feasibility of CAR T-cell therapy for 

In [19]:
## Extract structured Evidence and Relevance Filetering

def extract_evidence_table(question, articles, min_relevance_score = 3):
    """
    Extracts structured evidence parameters from each abstract and 
    filters out articles scoring below minimum relevance score
    """

    evidence_list = []

    instructions = """ 
        You are a biomedical data extraction assistant.
        Analyze the provided article abstract and extract the following structured details:
        1. Study objective
        2. Study type (e.g., Randomized Controlled Trial, Cohort study, Review, Systematic review, In vitro study, Not reported)
        3. Main finding
        4. Important limitation (Use "Not reported" if not stated in abstract)
        5. Relevance to research question
        6. Relevance score (Integer from 1 to 5, where 5 is highly relevant and 1 is completely irrelevant)

        Return ONLY a valid JSON object with the keys:
        "objective", "study-type", "main_finding", "limitation", "relevance_to_question", "relevance_score"
        Do not invent information.
        """

    for art in articles:
        user_prompt = f""" 
        Research question: {question}
        Article Title: {art['title']}
        Publication Year: {art['year']}
        Abstract: {art['abstract']}     
        """

        response = client.chat.completions.create(
            model = MODEL,
            messages = [
                {"role": "system", "content": instructions},
                {"role": "system", "content": user_prompt}
            ], response_format = {"type": "json_object"},
        )

        extracted = json.loads(response.choices[0].message.content)
        score = int(extracted.get("relevance_score", 3))

        # Exclude articles if relevance score <3
        if score >= min_relevance_score:
            evidence_list.append({
                "PMID": art["pmid"],
                "Article title": art["title"],
                "Publication Year": art["year"],
                "Study type": extracted.get("study_type", "Not reported"),
                "Study objective": extracted.get("objective", "Not reported"),
                "Main finding": extracted.get("main_finding", "Not reported"),
                "Important limitation": extracted.get("limitation", "Not reported"),
                "Relevance to research question": extracted.get("relevance_to_question", "Not reported"),
                "Relevance score": score,
                "Link": art["link"],
                "abstract": art["abstract"]
            })

    return evidence_list       


In [22]:
# reviewing abstacts 

def review_abstracts(question, query, articles):
    """
    Ask the model to review the retrieved abstracts.
    """ 

    article_text = ""

    # article formatting loop
    for number, article in enumerate(articles, start = 1):
        article_text += f""" 
        ARTICLE {number}
        PMID: {article["PMID"]}
        Title: {article["Article title"]}
        Year: {article["Publication Year"]}
        Link: {article["Link"]}
        Abstract: {article["abstract"]}
        """

    # defining system instructions/Prompt Engineering
    instructions = """
        You are assisting a biologist with a small PubMed abstract review.

        Use only the supplied abstracts. Do not invent findings, methods, or references. If information is missing, say so.

        Prepare a report containing:

        1. Research question
        2. PubMed query
        3. A short summary of each article
        4. Common findinds
        5. Important differences
        6. Limitations of the evidence
        7. References with PubMed IDs and links

        Place the relevant PMID after every article-level summary.
        Clearly state that the report is based only on abstracts.  
    """

    response = client.responses.create(
        model = MODEL,
        instructions = instructions,
        input = f""" 
                Research question: {question}
                PubMed query: {query}

            Retrieved articles:
            {article_text}
        """,
    )

    return response.output_text

In [23]:
## Main pipeline execution

def run_lr_agent():

    # Step1: User inputs
    print("=== Modified Literature-Review Agent ===")
    question = input("Enter your query: ")
    start_year = input("Enter earliest publication year (e.g. 2021): ") or "2021"
    max_articles_input = input("Enter maximum articles to review (1-5): ") or "5"
    maximum_articles = min(int(max_articles_input), 5) # Enforce limit <= 5

    # Step 2: Create PubMed Query
    print("\n[1/6] Generating PubMed query...")
    query = create_pubmed_query(question)
    print(f"Query: {query}")

    # Step 3: Search PubMed with Date Filter
    print(f"\n[2/6] Searching PubMed for articles from {start_year} onwards...")
    pmids = search_pubmed(query, start_year = start_year, maximum_results = maximum_articles)
    print(f"Retrieved PMIDs: {pmids}")

    if not pmids:
        print("No articles found matching the criteria.")
        return

    # Step 4: Fetch Abstracts
    print("\n[3/6] Fetching article abstracts...")
    raw_articles = fetch_abstracts(pmids)

    # Step 5: Extract Evidence & Apply Relevance Filter
    print("\n[4/6] Extracting evidence table and scoring relevance...")
    filtered_evidence = extract_evidence_table(question, raw_articles, min_relevance_score = 3)
    print (f"Retained {len(filtered_evidence)} articles after filtering.")


    if not filtered_evidence:
        print("All retreived articles were filtered out due to low relevance scores (<3).")
        return

    # Step 6: Save evidence table to CSV
    df_evidence = pd.DataFrame(filtered_evidence)
    # Exclude internal helper key 'abstract' from CSV export
    csv_df = df_evidence.drop(columns = ["abstract"])
    csv_df.to_csv("literature_review.csv", index = False)
    print("Saved evidence table to 'literature_review.csv'")

    # Step 7: Write & Save Final Literature Review
    print("\n[5/6] Wrtiting final literature review report...")
    review_markdown = review_abstracts(question, query, filtered_evidence)

    with open("literature_review.md", "w", encoding = "utf-8") as f:
        f.write(review_markdown)
    print("Saved written review to 'literature_review.md'")

    # Step 8: Display Output in Notebook
    print("\n[6/6] Final report Preview:")
    display(Markdown(review_markdown))

# Run the agent
if __name__ == "__main__":
    run_lr_agent()

=== Modified Literature-Review Agent ===

[1/6] Generating PubMed query...
Query: ("CD19"[MeSH Terms] OR CD19[Title/Abstract] OR "CD19 antigen"[Title/Abstract]) AND ("B-Cell Neoplasms"[MeSH Terms] OR "B-cell malignan*"[Title/Abstract] OR leukemia[Title/Abstract] OR lymphoma[Title/Abstract]) AND (target*[Title/Abstract] OR therapeut*[Title/Abstract] OR immunotherap*[Title/Abstract])

[2/6] Searching PubMed for articles from 2021 onwards...
Retrieved PMIDs: ['37748491', '35012754', '34445701', '40074511', '40771820']

[3/6] Fetching article abstracts...

[4/6] Extracting evidence table and scoring relevance...
Retained 4 articles after filtering.
Saved evidence table to 'literature_review.csv'

[5/6] Wrtiting final literature review report...
Saved written review to 'literature_review.md'

[6/6] Final report Preview:


# Abstract-Based Report: CD19 as a Target for B-Cell Malignancies

**Important note:** This report is based **only on the four supplied PubMed abstracts**. Full-text articles, supplementary data, and independent verification of the cited findings were not used.

## 1. Research question

**Is CD19 a better target for B-cell malignancies?**

Based on the supplied abstracts, CD19 appears to be a **well-validated and clinically effective target**, particularly for B-cell lymphomas, B-cell acute lymphoblastic leukemia (B-ALL), and some other B-cell malignancies. However, the abstracts do **not provide direct comparative evidence** showing that CD19 is superior to other B-cell targets.

## 2. PubMed query

```text
("CD19"[MeSH Terms] OR CD19[Title/Abstract] OR "CD19 antigen"[Title/Abstract])
AND
("B-Cell Neoplasms"[MeSH Terms] OR "B-cell malignan*"[Title/Abstract] OR leukemia[Title/Abstract] OR lymphoma[Title/Abstract])
AND
(target*[Title/Abstract] OR therapeut*[Title/Abstract] OR immunotherap*[Title/Abstract])
```

## 3. Short summary of each article

### Article 1: *CAR T-cell therapy for B-cell lymphoma* — 2022

This review describes anti-CD19 CAR T-cell therapy as a major advance for relapsed or refractory B-cell lymphomas. It reports that anti-CD19 CAR T-cell therapy has produced high response rates and durable responses in high-risk patients and has received FDA approval for several lymphoma subtypes. The abstract identifies three FDA-approved second-generation products—axicabtagene ciloleucel, tisagenlecleucel, and lisocabtagene maraleucel—and discusses their use in aggressive and indolent B-cell lymphomas. It also emphasizes that treatment failure can occur and that mechanisms of failure and post-CAR-T treatment strategies require further study.

**PMID: 35012754**  
https://pubmed.ncbi.nlm.nih.gov/35012754/

### Article 2: *CAR T-Cell Therapy in Hematological Malignancies* — 2021

This review discusses CAR T-cell therapy across hematological malignancies. It identifies CD19 as the most commonly used CAR T-cell target, particularly for lymphoma and acute lymphoblastic leukemia. The abstract states that CAR T-cell therapy has produced encouraging clinical results and led to approval of five CAR T-cell therapies. It also notes that treatment of other malignancies, such as acute myeloid leukemia, remains difficult and focuses on approaches to overcome CAR T-cell dysfunction and improve efficacy.

**PMID: 34445701**  
https://pubmed.ncbi.nlm.nih.gov/34445701/

### Article 3: *CARs for lymphoma* — 2024

This review describes CD19-targeting CAR T-cell therapy as an established treatment option for several B-cell malignancies, including diffuse large B-cell lymphoma, follicular lymphoma, mantle cell lymphoma, and chronic lymphocytic leukemia/small lymphocytic lymphoma. It reports robust and durable responses, including in heavily pretreated patients. The abstract also highlights important toxicities—cytokine release syndrome, immune effector cell neurotoxicity syndrome, cytopenias, infections, and secondary malignancies. Ongoing challenges include improving manufacturing, determining treatment sequencing, identifying optimal responders, and developing CARs directed at alternative targets or incorporating additional immune-stimulating functions.

**PMID: 40074511**  
https://pubmed.ncbi.nlm.nih.gov/40074511/

### Article 4: *Blinatumomab in pediatric B-acute lymphoblastic leukemia* — 2025

This review discusses blinatumomab, a bispecific T-cell engager, in pediatric relapsed or refractory B-ALL. The therapy targets CD19-expressing B-lineage leukemia cells through T-cell engagement and has demonstrated clinical benefits in this setting. The abstract identifies tumor burden, endogenous T-cell function, CD19 antigen loss, and lineage switch as factors influencing efficacy. It also reports risks including cytokine release syndrome, neurotoxicity, and infections. Although short-term efficacy and tolerability are described as favorable, the abstract states that long-term effects and improved dosing or combination strategies require further investigation.

**PMID: 40771820**  
https://pubmed.ncbi.nlm.nih.gov/40771820/

## 4. Common findings

Across the supplied abstracts:

- **CD19 is a clinically validated target** for multiple B-cell malignancies.
- CD19-directed therapies are used in:
  - B-cell lymphomas, including aggressive and indolent subtypes;
  - B-ALL;
  - chronic lymphocytic leukemia/small lymphocytic lymphoma, as reported in Article 3.
- **CAR T-cell therapy targeting CD19 can produce substantial and durable responses**, including in heavily pretreated or relapsed/refractory patients.
- CD19 is described as the **most commonly used CAR T-cell target** in hematological malignancies.
- CD19-directed therapies are associated with important toxicities, particularly:
  - cytokine release syndrome;
  - neurotoxicity or ICANS;
  - cytopenias;
  - infections.
- **Treatment resistance or failure remains possible.** The abstracts mention CD19 antigen loss, lineage switch, CAR T-cell dysfunction, and other mechanisms of failure.
- Further work is needed to improve:
  - treatment efficacy;
  - manufacturing;
  - patient selection;
  - sequencing with other therapies;
  - combination approaches;
  - management after treatment failure.

## 5. Important differences

| Aspect | Articles 1–3 | Article 4 |
|---|---|---|
| Main therapy | CD19-directed CAR T-cell therapy | Blinatumomab, a bispecific T-cell engager |
| Main disease focus | Primarily B-cell lymphomas; Article 2 also covers broader hematological malignancies | Pediatric relapsed/refractory B-ALL |
| Evidence emphasis | Response durability, regulatory approvals, treatment feasibility, CAR T-cell failure, and future CAR designs | Clinical efficacy determinants, antigen loss, lineage switch, dosing, complications, and long-term pediatric outcomes |
| Patient population | Mainly adults or broadly described hematological malignancy populations; exact populations are not consistently specified in the abstracts | Pediatric patients |
| Scope of target comparison | Article 3 mentions development of alternative CAR targets | The abstract does not compare CD19 directly with other antigen targets |
| Main limitations highlighted | CAR T-cell failure, manufacturing, toxicity, and responder prediction | CD19 antigen loss, lineage switch, toxicity, and uncertain long-term effects |

The articles also differ in the strength of their conclusions. Articles 1–3 describe CD19 CAR T-cell therapy as highly effective or transformative, whereas Article 4 illustrates that CD19-directed treatment can be affected by antigen loss and disease biology.

## 6. Limitations of the evidence

- **The evidence is based only on abstracts.** The abstracts do not provide the detailed study designs, patient numbers, response rates, confidence intervals, follow-up durations, or comparative statistics needed for a rigorous assessment.
- The supplied articles are **reviews**, rather than clearly identified head-to-head clinical trials comparing CD19 with alternative targets.
- The abstracts do not establish that CD19 is **superior** to other targets such as those directed against alternative B-cell antigens.
- The reported benefits may reflect different diseases, treatment settings, patient populations, and therapeutic platforms, making direct comparison difficult.
- The abstracts do not consistently distinguish outcomes by:
  - specific malignancy subtype;
  - treatment line;
  - disease burden;
  - prior therapies;
  - CAR construct;
  - manufacturing process;
  - follow-up duration.
- The abstracts emphasize clinical success but do not provide enough information to quantify:
  - response rates;
  - relapse rates;
  - duration of response;
  - overall survival;
  - toxicity frequency;
  - quality-of-life outcomes.
- CD19 antigen loss and lineage switch indicate that **CD19 is not an universally reliable target**, particularly when tumor cells lose or alter CD19 expression.
- Long-term risks and outcomes, including secondary malignancies and pediatric long-term effects, remain incompletely characterized in the supplied abstracts.
- The abstracts do not allow assessment of cost, accessibility, manufacturing time, or comparative treatment burden.

## Overall conclusion

The supplied abstracts support the conclusion that **CD19 is one of the best-established and most clinically successful targets for B-cell malignancies**, especially for relapsed or refractory B-cell lymphoma and B-ALL. Its strengths include broad clinical applicability, durable responses in some patients, and use across multiple immunotherapy platforms.

However, the available abstracts **do not demonstrate that CD19 is definitively better than alternative targets**. CD19-directed treatment is limited by toxicities, treatment failure, CD19 antigen loss, lineage switch, and other forms of resistance. A stronger answer to the comparative question would require full-text evidence and direct clinical comparisons with alternative B-cell targets.

## 7. References

1. **CAR T-cell therapy for B-cell lymphoma.** 2022. PMID: **35012754**.  
   https://pubmed.ncbi.nlm.nih.gov/35012754/

2. **CAR T-Cell Therapy in Hematological Malignancies.** 2021. PMID: **34445701**.  
   https://pubmed.ncbi.nlm.nih.gov/34445701/

3. **CARs for lymphoma.** 2024. PMID: **40074511**.  
   https://pubmed.ncbi.nlm.nih.gov/40074511/

4. **Blinatumomab in pediatric B-acute lymphoblastic leukemia.** 2025. PMID: **40771820**.  
   https://pubmed.ncbi.nlm.nih.gov/40771820/